In [ ]:
import pandas as pd
import geopandas as gpd
import sys

reeds_path = '' # User should specify path to ReEDS repository here
sys.path.append(reeds_path)
import reeds

Especially on Linux, gdxpds should be imported before pandas to avoid a library conflict. Also make sure your GAMS directory is listed in LD_LIBRARY_PATH.


In [4]:
state_fips_to_abbrev_map = {
    '01': 'AL',
    '02': 'AK',
    '04': 'AZ',
    '05':'AR',
    '06':'CA',
    '08':'CO',
    '09':'CT',
    '10':'DE',
    '11':'DC',
    '12':'FL',
    '13':'GA',
    '15':'HI',
    '16':'ID',
    '17':'IL',
    '18':'IN',
    '19':'IA',
    '20':'KS',
    '21':'KY',
    '22':'LA',
    '23':'ME',
    '24':'MD',
    '25':'MA',
    '26':'MI',
    '27':'MN',
    '28':'MS',
    '29':'MO',
    '30':'MT',
    '31':'NE',
    '32':'NV',
    '33':'NH',
    '34':'NJ',
    '35':'NM',
    '36':'NY',
    '37':'NC',
    '38':'ND',
    '39':'OH',
    '40':'OK',
    '41':'OR',
    '42':'PA',
    '44':'RI',
    '45':'SC',
    '46':'SD',
    '47':'TN',
    '48':'TX',
    '49':'UT',
    '50':'VT',
    '51':'VA',
    '53':'WA',
    '54':'WV',
    '55':'WI',
    '56':'WY',
}

In [5]:
gasreg_state_map = {
    'Northwest': ['OR', 'WA'],
    'California': ['CA'],
    'New England': ['ME', 'NH', 'VT', 'MA', 'CT', 'RI'],
    'Mid-Atlantic': ['NY', 'NJ', 'PA'],
    'South Atlantic': ['DC', 'DE', 'MD', 'NC', 'SC', 'GA', 'FL', 'WV', 'VA'],
    'East North Central': ['WI', 'IL', 'IN', 'MI', 'OH'],
    'Mountain': ['MT', 'ID', 'WY', 'CO', 'UT', 'NV'],
    'Southwest': ['AZ', 'NM'],
    'West South Central': ['TX', 'OK', 'AR', 'LA'],
    'East South Central': ['KY', 'TN', 'MS', 'AL'],
    'West North Central': ['ND', 'MN', 'SD', 'NE', 'IA', 'KS', 'MO']
}

state_region_map = {}
for region, state_list in gasreg_state_map.items():
    for state in state_list:
        state_region_map[state] = region

In [6]:
county_geo = reeds.spatial.get_map('county', source='tiger')

hub_locations = pd.read_excel('//nrelnas01/ReEDS/FY26_NatGas_KO/HE_VelSuite_GasIndices.xlsx')
hub_locations = gpd.GeoDataFrame(
    hub_locations,
    geometry=gpd.points_from_xy(hub_locations.Longitude, hub_locations.Latitude),
    crs='EPSG:4326'
)
hub_locations = hub_locations.to_crs(county_geo.crs)

county_geo['STCODE'] = county_geo['STATEFP'].map(state_fips_to_abbrev_map)
county_geo['gasreg'] = county_geo.STCODE.map(state_region_map)
gasreg_geo = county_geo.dissolve('gasreg')
hub_locations = gpd.sjoin(hub_locations, gasreg_geo[['geometry']])
hub_gasreg_map = dict(zip(hub_locations['Price_Hub'], hub_locations['gasreg']))

hub_gasreg_map

{'Hitachi Energy Barnett': 'West South Central',
 'Hitachi Energy California - North': 'California',
 'Hitachi Energy California - South': 'California',
 'Hitachi Energy Chicago Metro': 'East North Central',
 'Hitachi Energy Green River': 'Mountain',
 'Hitachi Energy Gulf Coast ELA': 'West South Central',
 'Hitachi Energy Gulf Coast ETX': 'West South Central',
 'Hitachi Energy Gulf Coast STX': 'West South Central',
 'Hitachi Energy Gulf Coast WLA': 'West South Central',
 'Hitachi Energy Haynesville': 'West South Central',
 'Hitachi Energy Henry Hub': 'West South Central',
 'Hitachi Energy Marcellus - Central': 'Mid-Atlantic',
 'Hitachi Energy Marcellus - Lower': 'South Atlantic',
 'Hitachi Energy Marcellus - Upper': 'Mid-Atlantic',
 'Hitachi Energy Michigan': 'East North Central',
 'Hitachi Energy Michigan/Ontario': 'East North Central',
 'Hitachi Energy Mid-Atlantic': 'South Atlantic',
 'Hitachi Energy Midcontinent - Central': 'West North Central',
 'Hitachi Energy Midcontinent - East